# Clustering with K-Means — Solutions

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/rework/solutions/05_Clustering_KMeans_solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏱️ ~75 min · Day 2

**⚠️ Important:** Try the exercises in `demos/05_Clustering_KMeans.ipynb` yourself first. The struggle is where the learning happens.

These are *reference* solutions with short explanations. Your own working solution may look different and still be correct.

In [ ]:
# === COURSE SETUP — run this cell first! ===
%pip install -q numpy pandas matplotlib seaborn scikit-learn

import os, urllib.request
DATA_URL = "https://raw.githubusercontent.com/LuWidme/uk259/rework/datasets/"
for folder in ("datasets", os.path.join("..", "datasets")):
    os.makedirs(folder, exist_ok=True)
    for fname in ['titanic.csv', 'melb_data.csv', 'Company_data.csv']:
        path = os.path.join(folder, fname)
        if not os.path.exists(path):
            urllib.request.urlretrieve(DATA_URL + fname, path)
print("Setup complete — you are ready to go!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import pairwise_distances
np.random.seed(42)

# Synthetic data (same idea as in the demo)
features, true_labels = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Customer data: Annual Income (20k-100k) and Spending Score (1-100)
customer_raw, _ = make_blobs(n_samples=200, centers=5, cluster_std=1.2, random_state=42)
customer_data = np.column_stack([
    np.clip(np.interp(customer_raw[:,0], (customer_raw[:,0].min(), customer_raw[:,0].max()), (20, 100)), 20, 100),
    np.clip(np.interp(customer_raw[:,1], (customer_raw[:,1].min(), customer_raw[:,1].max()), (1, 100)), 1, 100),
])
print('Data ready:', features.shape, customer_data.shape)

## Exercise 1: Implement the Elbow Method

In [ ]:
k_values = range(1, 11)
inertias = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.show()
print('The elbow is around K=4 — adding more clusters barely lowers inertia.')

## Exercise 2: Customer Segmentation

Main exercise: find the optimal K with the elbow method, then cluster and visualise.

In [ ]:
# Elbow on customer data
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(customer_data)
    inertias.append(km.inertia_)
plt.figure(figsize=(8,5))
plt.plot(range(2,11), inertias, 'go-')
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow — customers')
plt.show()

In [ ]:
optimal_k = 5   # chosen from the elbow plot
kmeans_customer = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_labels = kmeans_customer.fit_predict(customer_data)
customer_centers = kmeans_customer.cluster_centers_

plt.figure(figsize=(10, 7))
plt.scatter(customer_data[:,0], customer_data[:,1], c=customer_labels, cmap='viridis', alpha=0.7)
plt.scatter(customer_centers[:,0], customer_centers[:,1], c='red', marker='X', s=200, label='Centers')
plt.xlabel('Annual Income (k$)'); plt.ylabel('Spending Score')
plt.title('Customer Segments'); plt.legend(); plt.show()

## Bonus: K-Means from Scratch

### Task 1: Initialize Cluster Centers

In [ ]:
def initialize_centers(data, k):
    n_samples = data.shape[0]
    indices = np.random.choice(n_samples, k, replace=False)
    return data[indices]

centers = initialize_centers(features, 4)
print('Initial centers:\n', centers)

### Task 2: Assign Points to Nearest Cluster

In [ ]:
def assign_clusters(data, centers):
    distances = pairwise_distances(data, centers)
    return np.argmin(distances, axis=1)

labels = assign_clusters(features, centers)
print('First 10 labels:', labels[:10])

### Task 3: Update Cluster Centers

In [ ]:
def update_centers(data, labels, k):
    n_features = data.shape[1]
    new_centers = np.zeros((k, n_features))
    for i in range(k):
        points = data[labels == i]
        if len(points) > 0:
            new_centers[i] = points.mean(axis=0)
    return new_centers

centers = update_centers(features, labels, 4)
print('Updated centers:\n', centers)

### Task 4: Complete K-Means Algorithm

In [ ]:
def k_means(data, k, max_iter=100, tolerance=1e-4):
    centers = initialize_centers(data, k)
    for _ in range(max_iter):
        labels = assign_clusters(data, centers)
        new_centers = update_centers(data, labels, k)
        if np.linalg.norm(new_centers - centers) < tolerance:
            break
        centers = new_centers
    return labels, centers

labels, centers = k_means(features, 4)
plt.figure(figsize=(9,6))
plt.scatter(features[:,0], features[:,1], c=labels, cmap='viridis', alpha=0.6)
plt.scatter(centers[:,0], centers[:,1], c='red', marker='X', s=200)
plt.title('K-Means from scratch'); plt.show()